In [1]:
# Training and Saving Final Models

import pandas as pd

import joblib

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC

from sklearn.pipeline import Pipeline

def LS1():

    print("-" * 40)
    print("LinearSVC (1) - Final Model")

    df = pd.read_csv("vsrs_400_classified.csv")

    vsrs = df["vsrs"]
    classification = df["classification"]

    # Create model
    model = Pipeline([
        ("text_to_numbers", TfidfVectorizer(
            ngram_range=(1,2))
        ),
        ("classifier", LinearSVC(
            class_weight="balanced"))
    ])

    # Train using all the manually classified data (400-sample)
    model.fit(vsrs, classification)

    # Save model
    joblib.dump(model, "LS1.pkl")

    print("Model saved as LS1.pkl")


def LS2_1():

    print("-" * 40)
    print("LinearSVC (2.1 - Relevance) - Final Model")

    df = pd.read_csv("vsrs_400_relevance.csv")

    vsrs = df["vsrs"]
    relevance = df["relevance"]

    # Create model
    model = Pipeline([
        ("text_to_numbers", TfidfVectorizer(
            ngram_range=(1,2))
        ),
        ("classifier", LinearSVC(
            class_weight="balanced"))
    ])

    # Train using all the manually classified data (400-sample)
    model.fit(vsrs, relevance)

    # Save model
    joblib.dump(model, "LS2_1.pkl")

    print("Model saved as LS2_1.pkl")


def LS2_2():

    print("-" * 40)
    print("LinearSVC (2.2 - Legality) - Final Model")

    df = pd.read_csv("vsrs_400_legality.csv")

    vsrs = df["vsrs"]
    legality = df["legality"]

    # Create model
    model = Pipeline([
        ("text_to_numbers", TfidfVectorizer(
            ngram_range=(1,2))
        ),
        ("classifier", LinearSVC(
            class_weight="balanced"))
    ])

    # Train using all the manually classified data (400-dataset)
    model.fit(vsrs, legality)

    # Save model
    joblib.dump(model, "LS2_2.pkl")

    print("Model saved as LS2_2.pkl")

LS1()   # Single-stage
LS2_1() # Dual-stage - Stage 1
LS2_2() # Dual-stage - Stage 2

----------------------------------------
LinearSVC (1) - Final Model
Model saved as LS1.pkl
----------------------------------------
LinearSVC (2.1 - Relevance) - Final Model
Model saved as LS2_1.pkl
----------------------------------------
LinearSVC (2.2 - Legality) - Final Model
Model saved as LS2_2.pkl


In [3]:
# NLP Prediction on the 1240-dataset

import pandas as pd

import joblib

# Load NLP models
ls1 = joblib.load("LS1.pkl")
ls2_1 = joblib.load("LS2_1.pkl")
ls2_2 = joblib.load("LS2_2.pkl")

# Load dataset for NLP prediction
df = pd.read_csv("vsrs_1240.csv")
print("VSRS 1240 dataset")
print("Number of rows:", len(df))

# Single-stage model
df["LS1 Classification"] = ls1.predict(df["vsrs"])

# Dual-stage model
relevance = ls2_1.predict(df["vsrs"]) # Stage 1: Relevance
df.loc[relevance == "no", "LS2 Classification"] = "blue"
relevant = relevance == "yes" # Select VSRS classified as relevant
legality = ls2_2.predict(df.loc[relevant, "vsrs"]) # Stage 2: Legality
df.loc[relevant, "LS2 Classification"] = legality
df.loc[df["LS2 Classification"] == "yes", "LS2 Classification"] = "green"
df.loc[df["LS2 Classification"] == "no", "LS2 Classification"] = "amber"

# Save NLP-predicted VSRS classifications
df.to_csv("vsrs_1240_predicted.csv", index=False)

print("-"*40)

# Determininhg the number of jobs in each category for each model
ls1_green = (df["LS1 Classification"] == "green").sum()
ls1_amber = (df["LS1 Classification"] == "amber").sum()
ls1_blue = (df["LS1 Classification"] == "blue").sum()

ls2_green = (df["LS2 Classification"] == "green").sum()
ls2_amber = (df["LS2 Classification"] == "amber").sum()
ls2_blue = (df["LS2 Classification"] == "blue").sum()

print("LS1 Green:", ls1_green)
print("LS1 Amber:", ls1_amber)
print("LS1 Blue:", ls1_blue)
print("-"*40)
print("LS2 Green:", ls2_green)
print("LS2 Amber:", ls2_amber)
print("LS2 Blue:", ls2_blue)

VSRS 1240 dataset
Number of rows: 1240
----------------------------------------
LS1 Green: 672
LS1 Amber: 384
LS1 Blue: 184
----------------------------------------
LS2 Green: 675
LS2 Amber: 384
LS2 Blue: 181


In [4]:
# Generating CSVs for analysis of NLP models' performance

import pandas as pd

df = pd.read_csv("vsrs_1240_predicted.csv")
comparison = df[["vsrs", "LS1 Classification", "LS2 Classification"]]
comparison.to_csv("vsrs_1240_comparison.csv", index=False) # only VSRS and Classification columns

different = comparison[comparison["LS1 Classification"] != comparison["LS2 Classification"]]
different.to_csv("vsrs_1240_difference.csv", index=False) # only rows with differing Classifications
print("Datapoints with different classifications between LS1 and LS2:",len(different),"\n"+"-"*80)
print(different)

Datapoints with different classifications between LS1 and LS2: 5 
--------------------------------------------------------------------------------
                                                  vsrs LS1 Classification  \
10   The legal right to live and work in the UK on ...              green   
635  DESCRIPTION Job summary Sponsored Display is A...               blue   
786  export laws without sponsorship for an export ...               blue   
924  Applicants for employment in the US/Canada mus...              amber   
954  export laws without sponsorship for an export ...               blue   

    LS2 Classification  
10               amber  
635              green  
786              green  
924              green  
954              green  


In [ ]:
# Create a random (reproducible) sample of n rows for comparison (VSRS and Classification columns only)

import pandas as pd

df = pd.read_csv("vsrs_1240_comparison.csv")

n = 50

sample = df.sample(n=n, random_state=42)
sample.to_csv("vsrs_1240_comparison_sample.csv", index=False)

print("Saved", n, "random rows to vsrs_1240_comparison_sample.csv")

Saved 50 random rows to vsrs_1240_comparison_sample.csv
